# Build Complete Vulnerability Library

Download NVD data from 2002-2025 and build enriched cache with CVD analytics.

In [ ]:
from vulnflow import VulnDataset
from vulnflow_data import (
    download_year_range,
    download_modified,
    build_cache,
    list_downloaded_files,
    get_available_years,
    DEFAULT_TRANSFORMS,
)
from pathlib import Path
import pandas as pd

## Configuration

Choose your year range. Start small for testing, expand for production.

In [ ]:
# Available years: 2002-2025
print(f"Available years: {get_available_years()}")

# Choose range (start with recent years for faster testing)
START_YEAR = 2023  # Change to 2002 for complete library
END_YEAR = 2025

print(f"\nBuilding library for {START_YEAR}-{END_YEAR}")

## Step 1: Download NVD data

Downloads ~1-5 MB per year. Full library (2002-2025) is ~100-150 MB.

In [ ]:
# Download year range
paths = download_year_range(START_YEAR, END_YEAR)

## Step 2: Build enriched cache

Applies all transforms: flatten (CVSS, CWE, CPE), CVD analytics, event inference.

**Processing time:**
- 2023-2025 (~100K CVEs): ~30 seconds
- 2002-2025 (~300K CVEs): ~90 seconds

In [ ]:
# Build cache with all transforms
cache_path = build_cache(
    sources=paths,
    output_path="../data/nvd_library.parquet",
    transforms=DEFAULT_TRANSFORMS,
)

print(f"\nCache saved to: {cache_path}")

## Step 3: Load and explore

Cache loads in ~1-3 seconds regardless of size.

In [ ]:
# Load cache to pandas
df = VulnDataset.from_parquet(cache_path).to_pandas()

print(f"Library size: {len(df):,} CVEs")
print(f"Columns: {len(df.columns)}")
print(f"Memory: {df.memory_usage(deep=True).sum() / 1024 / 1024:.1f} MB")

In [ ]:
df.head()

## Explore the library

In [ ]:
# CVEs by year
df['nvd_published_at'] = pd.to_datetime(df['nvd_published_at'])
df['year'] = df['nvd_published_at'].dt.year
df['year'].value_counts().sort_index().plot(kind='bar', figsize=(12, 4), title='CVEs by Year')

In [ ]:
# Severity distribution
df['cvss_base_severity'].value_counts()

In [ ]:
# Top CWEs
df['cwe_id'].value_counts().head(10)

In [ ]:
# Top vendors
df['cpe_vendor'].value_counts().head(10)

## CVD Analytics

In [ ]:
# Zero-day exploits
zero_days = df[df['is_zero_day_exploit'] == True]
print(f"Zero-day exploits: {len(zero_days)}")
zero_days[['cve_id', 'cvss_base_score', 'year', 'description']].head(10)

In [ ]:
# Fix lag analysis
df['fix_lag_days'].describe()

In [ ]:
# CVD states
df['cvd_state'].value_counts().head(10)

## Incremental updates

Keep library current with Modified feed.

In [ ]:
# Download Modified feed
modified_path = download_modified()

# Build delta cache
delta_cache = build_cache(
    sources=[modified_path],
    output_path="../data/nvd_delta.parquet",
    transforms=DEFAULT_TRANSFORMS,
)

In [ ]:
# Merge library + delta
df_current = VulnDataset.from_parquet([
    "../data/nvd_library.parquet",
    "../data/nvd_delta.parquet",
]).to_pandas()

print(f"Current library: {len(df_current):,} CVEs")
print(f"Added: {len(df_current) - len(df)} CVEs from delta")

## Downloaded files

In [ ]:
# Show all downloaded files
files = list_downloaded_files()
for name, path in sorted(files.items()):
    size_mb = path.stat().st_size / 1024 / 1024
    print(f"{name}: {size_mb:.1f} MB")

## Next steps

**For production:**
1. Set `START_YEAR = 2002` to build complete library (~300K CVEs)
2. Run this notebook once to build cache
3. Use notebook 01 or 02 for daily analysis
4. Re-run delta merge weekly to stay current

**Scheduled updates:**
```python
# Daily script
from vulnflow_data import download_modified, build_cache
from vulnflow import VulnDataset

# Update delta
modified = download_modified(force=True)
build_cache([modified], output_path="data/nvd_delta.parquet")

# Merge and analyze
df = VulnDataset.from_parquet([
    "data/nvd_library.parquet",
    "data/nvd_delta.parquet",
]).to_pandas()
```